# 📊 M1MLOP — Jour 3 : Monitoring et Gouvernance
**École IT — Master 1 — Unité 3 BigData — Semaine 11**  
**Amadou MAIGA**

---

## 🗂️ Table des matières
1. [Objectifs et contexte](#1)
2. [Pourquoi monitorer les modèles ML ?](#2)
3. [Types de drift](#3)
4. [TP 3.1 — Drift Detection (KS Test)](#4)
5. [Métriques de monitoring ML](#5)
6. [TP 3.2 — Métriques Prometheus](#6)
7. [A/B Testing pour modèles ML](#7)
8. [TP 3.3 — A/B Test (Chi-Square)](#8)
9. [Gouvernance — Explicabilité (SHAP)](#9)
10. [TP 3.4 — Fairness & Détection de biais](#10)
11. [Résumé et bilan du Jour 3](#11)

---

> 📌 **Prérequis** : Jour 1 (MLflow) + Jour 2 (Pipelines, Docker)  
> ⏱️ **Durée estimée** : 7h (matinée + après-midi)

<a id='1'></a>
## 1. 🎯 Objectifs du Jour 3

À la fin de cette journée, tu seras capable de :

- ✅ Détecter les **drifts** de données et de modèles en production
- ✅ Mettre en place l'**observabilité** : logs, métriques, alertes
- ✅ Implémenter des **tests A/B** et déploiements graduels
- ✅ Garantir la **gouvernance** et la conformité des modèles (RGPD, explicabilité)
- ✅ Détecter les **biais** dans les prédictions d'un modèle

### 🔗 Lien avec les jours précédents

```
Jour 1 : MLflow (tracking, registry)
    ↓
Jour 2 : Pipelines Airflow + Docker + K8s
    ↓
Jour 3 : Monitoring (le modèle est en prod → comment le surveiller ?)
```

<a id='2'></a>
## 2. ⚠️ Pourquoi Monitorer les Modèles ML ?

Contrairement au **software classique** (le code ne change pas → le comportement reste identique), les **modèles ML dégradent naturellement** en production.

> 💰 **Amazon (Recommandations)** : Un modèle de recommandation avec une dégradation **invisible de 0.1%** de précision coûte **10M$ par an** en ventes perdues. Sans monitoring, personne ne le remarque pendant des mois.

### Software classique vs Modèle ML

```
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  SOFTWARE CLASSIQUE          MODÈLE ML                     │
│  ─────────────────           ────────                      │
│  Code statique               Données qui changent          │
│  Bugs = erreurs claires      Dégradation graduelle         │
│  Fix : patch le code         Fix : retraining              │
│  Surveillance : logs         Surveillance : métriques ML   │
│                              + distribution des données    │
└─────────────────────────────────────────────────────────────┘
```

### Seuils d'alerte typiques en production

| Métrique | Seuil WARNING | Seuil CRITICAL | Action |
|---|---|---|---|
| Accuracy | < 90% | < 85% | Alerte équipe |
| Latence | > 200ms | > 500ms | Scaling K8s |
| Data Drift (p-value) | < 0.10 | < 0.05 | Retraining |
| Erreur rate | > 1% | > 5% | Rollback |

<a id='3'></a>
## 3. 🌊 Types de Drift

```
┌─────────────────────────────────────────────────────────────┐
│                    TYPES DE DRIFT ML                        │
├──────────────────┬──────────────────────────────────────────┤
│ DATA DRIFT       │ Les features d'entrée changent           │
│                  │ Ex: clients urbains → clients ruraux     │
│                  │ Détection : KS test sur distributions    │
├──────────────────┼──────────────────────────────────────────┤
│ CONCEPT DRIFT    │ La relation features→target change       │
│                  │ Ex: salaires augmentent de 20%           │
│                  │ Détection : erreur moyenne des preds     │
├──────────────────┼──────────────────────────────────────────┤
│ COVARIATE DRIFT  │ Distribution des features seulement      │
├──────────────────┼──────────────────────────────────────────┤
│ LABEL DRIFT      │ Distribution du target change            │
└──────────────────┴──────────────────────────────────────────┘
```

> 🏥 **Diagnostic médical** : Un modèle formé sur les patients de 2020 peut dégrader sur les patients de 2024 (nouvelles variantes, démographie changeante). La drift detection est **critique pour la santé**.

> 🚗 **Tesla (Autonomie)** : Les modèles de conduite autonome dégradent avec les **saisons** (la neige change les patterns visuels). Tesla re-entraîne les modèles en continu basé sur le monitoring real-time.

In [ ]:
# ============================================================
# 📦 INSTALLATION DES DÉPENDANCES JOUR 3
# ============================================================
import sys

!{sys.executable} -m pip install scikit-learn pandas numpy matplotlib scipy prometheus-client --quiet

print("✅ Dépendances installées !")
print("ℹ️  Pour SHAP (optionnel) : pip install shap")

In [ ]:
# ============================================================
# 📚 IMPORTS GLOBAUX
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import time
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import ks_2samp, chi2_contingency
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✅ Imports OK")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")
print(f"   SciPy  : importé")

<a id='4'></a>
## 4. 🧪 TP 3.1 — Drift Detection avec le Test KS

### Objectif
Implémenter la détection de drift avec le **test de Kolmogorov-Smirnov** et tester 3 niveaux de drift différents.

### Le test KS (Kolmogorov-Smirnov)

Le test KS compare **deux distributions** et répond à la question :  
*"Ces deux échantillons viennent-ils de la même distribution ?"*

| p-value | Interprétation | Action |
|---|---|---|
| > 0.10 | Distributions similaires | ✅ NORMAL |
| 0.05 – 0.10 | Légère différence | ⚠️ WARNING |
| 0.01 – 0.05 | Drift confirmé | 🔴 ALERTE |
| < 0.01 | Drift critique | 🚨 CRITICAL → Retraining |

> 📊 **Spotify** : Spotify utilise drift detection pour ses modèles de recommandation musicale. Chaque mois, le goût des utilisateurs change → drift detection déclenche un retraining automatique.

In [ ]:
# ============================================================
# TP 3.1 — FONCTION DE DRIFT DETECTION
# ============================================================

def detect_drift(training_data: np.ndarray,
                 production_data: np.ndarray,
                 feature_name: str = "feature",
                 threshold: float = 0.05) -> dict:
    """
    Détecte si la distribution de production diffère du training.
    Utilise le test de Kolmogorov-Smirnov (KS test).

    Args:
        training_data   : Features du dataset de training (baseline)
        production_data : Features actuelles en production
        feature_name    : Nom de la feature (pour les logs)
        threshold       : p-value seuil pour déclarer un drift (défaut 0.05)

    Returns:
        dict avec drift_detected, p_value, severity, recommendation
    """
    statistic, p_value = ks_2samp(training_data, production_data)

    if p_value < 0.01:
        severity      = "CRITICAL 🚨"
        recommendation = "Retraining immédiat requis"
        color          = "red"
    elif p_value < 0.05:
        severity      = "WARNING 🔴"
        recommendation = "Planifier un retraining"
        color          = "orange"
    elif p_value < 0.10:
        severity      = "WATCH ⚠️"
        recommendation = "Surveiller de près"
        color          = "yellow"
    else:
        severity      = "NORMAL ✅"
        recommendation = "Aucune action requise"
        color          = "green"

    return {
        "feature"        : feature_name,
        "drift_detected" : p_value < threshold,
        "p_value"        : round(p_value, 6),
        "ks_statistic"   : round(float(statistic), 4),
        "severity"       : severity,
        "recommendation" : recommendation,
        "color"          : color,
        "train_mean"     : round(float(np.mean(training_data)), 4),
        "prod_mean"      : round(float(np.mean(production_data)), 4),
        "mean_shift"     : round(float(np.mean(production_data) - np.mean(training_data)), 4)
    }

print("✅ Fonction detect_drift définie !")

In [ ]:
# ============================================================
# TEST SUR 3 NIVEAUX DE DRIFT
# ============================================================

np.random.seed(SEED)

# Données de training (baseline)
train_ages = np.random.normal(loc=35, scale=8, size=1000)   # Moyenne 35 ans

# 3 scénarios de production
scenarios = [
    {
        "name"  : "Drift FAIBLE",
        "data"  : np.random.normal(loc=36, scale=8, size=500),  # +1 an de décalage
        "desc"  : "Légère évolution démographique"
    },
    {
        "name"  : "Drift MOYEN",
        "data"  : np.random.normal(loc=42, scale=8, size=500),  # +7 ans de décalage
        "desc"  : "Nouveaux clients plus âgés"
    },
    {
        "name"  : "Drift CRITIQUE",
        "data"  : np.random.normal(loc=55, scale=10, size=500), # +20 ans de décalage
        "desc"  : "Population totalement différente"
    }
]

print("=" * 70)
print("          📊 ANALYSE DE DRIFT — 3 SCÉNARIOS")
print("=" * 70)
print(f"  Baseline : âge moyen = {train_ages.mean():.1f} ans (N={len(train_ages)})")
print()

drift_results = []
for sc in scenarios:
    result = detect_drift(train_ages, sc["data"], feature_name="age")
    result["scenario"] = sc["name"]
    result["description"] = sc["desc"]
    drift_results.append(result)

    print(f"  📌 {sc['name']:20s} | {sc['desc']}")
    print(f"     Moyenne prod     : {result['prod_mean']:.1f} ans (shift: +{result['mean_shift']:.1f})")
    print(f"     p-value          : {result['p_value']:.6f}")
    print(f"     Drift détecté    : {'OUI' if result['drift_detected'] else 'NON'}")
    print(f"     Sévérité         : {result['severity']}")
    print(f"     Recommandation   : {result['recommendation']}")
    print()

print("=" * 70)

In [ ]:
# ============================================================
# VISUALISATION DES 3 NIVEAUX DE DRIFT
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('🌊 Visualisation du Data Drift — TP 3.1', fontsize=14, fontweight='bold')

colors_map = {"NORMAL ✅": "green", "WATCH ⚠️": "goldenrod",
              "WARNING 🔴": "orange", "CRITICAL 🚨": "red"}

for i, (sc, res) in enumerate(zip(scenarios, drift_results)):
    ax = axes[i]
    color = colors_map.get(res["severity"], "gray")

    ax.hist(train_ages, bins=30, alpha=0.6, color='steelblue', label='Training (baseline)', density=True)
    ax.hist(sc["data"],  bins=30, alpha=0.6, color=color,      label='Production', density=True)

    ax.axvline(train_ages.mean(), color='steelblue', linestyle='--', linewidth=2, label=f'Mean train: {train_ages.mean():.1f}')
    ax.axvline(sc["data"].mean(), color=color,       linestyle='--', linewidth=2, label=f'Mean prod: {sc["data"].mean():.1f}')

    ax.set_title(f'{sc["name"]}\np-value={res["p_value"]:.4f} | {res["severity"]}', fontsize=10)
    ax.set_xlabel('Âge')
    ax.set_ylabel('Densité')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    border_color = colors_map.get(res["severity"], "gray")
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(2.5)

plt.tight_layout()
plt.savefig('drift_detection_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Graphique sauvegardé : drift_detection_visualization.png")

In [ ]:
# ============================================================
# DRIFT DETECTION SUR TOUTES LES FEATURES DU DATASET IRIS
# ============================================================

iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names

# Simuler un dataset de training et un de production avec drift
X_train, X_prod_base = train_test_split(X, test_size=0.5, random_state=SEED)

# Ajouter du drift sur certaines features
np.random.seed(SEED)
X_prod_drifted = X_prod_base.copy()
X_prod_drifted[:, 0] += np.random.normal(0.5, 0.3, len(X_prod_drifted))   # Drift sur sepal length
X_prod_drifted[:, 2] += np.random.normal(1.5, 0.5, len(X_prod_drifted))   # Drift fort sur petal length

print("📊 Drift Detection — Dataset Iris (toutes les features)")
print("=" * 70)
print(f"  {'Feature':30s} {'p-value':>10s} {'Drift':>8s} {'Sévérité':>15s}")
print("-" * 70)

feature_drift_results = []
for i, fname in enumerate(feature_names):
    res = detect_drift(X_train[:, i], X_prod_drifted[:, i], feature_name=fname)
    feature_drift_results.append(res)
    drift_str = "OUI 🔴" if res["drift_detected"] else "non ✅"
    print(f"  {fname:30s} {res['p_value']:>10.6f} {drift_str:>8s} {res['severity']:>15s}")

print("=" * 70)
n_drift = sum(1 for r in feature_drift_results if r["drift_detected"])
print(f"  ⚠️  {n_drift}/{len(feature_names)} features en drift")
if n_drift > 0:
    print("  🔁 Action recommandée : planifier un retraining")

<a id='5'></a>
## 5. 📈 Métriques de Monitoring ML

### L'observabilité en MLOps

L'observabilité = capacité à comprendre l'état interne du système à partir de ses **outputs**.

```
┌────────────────────────────────────────────────────────┐
│              STACK DE MONITORING ML                    │
│                                                        │
│  API Flask/FastAPI                                     │
│     │  expose métriques sur /metrics                  │
│     ↓                                                  │
│  Prometheus (scrape toutes les 30s)                    │
│     │  stocke les séries temporelles                   │
│     ↓                                                  │
│  Grafana (dashboards visuels)                          │
│     │  visualise accuracy, latency, drift              │
│     ↓                                                  │
│  AlertManager (alertes Slack/email)                    │
│     → si accuracy < 85% → alerte équipe               │
└────────────────────────────────────────────────────────┘
```

### 3 types de métriques Prometheus

| Type | Usage ML | Exemple |
|---|---|---|
| **Counter** | Compteurs qui ne font qu'augmenter | Nombre total de prédictions |
| **Gauge** | Valeurs instantanées (up/down) | Accuracy courante, pods actifs |
| **Histogram** | Distribution de valeurs | Latence des prédictions |

> 🔧 **LinkedIn** : LinkedIn collecte **10M+ métriques par seconde** avec Prometheus. Les modèles de ranking sont monitorés en temps réel.

<a id='6'></a>
## 6. 🧪 TP 3.2 — Métriques Prometheus

### Objectif
Exporter des **métriques ML** en format Prometheus et simuler 100 prédictions avec monitoring.

In [ ]:
# ============================================================
# TP 3.2 — MÉTRIQUES PROMETHEUS
# ============================================================

from prometheus_client import Counter, Gauge, Histogram, CollectorRegistry, generate_latest
import threading

# Créer un registre isolé (pour le notebook)
registry = CollectorRegistry()

# ---- Définir les métriques ----

# Counter : nombre total de prédictions (ne fait qu'augmenter)
predictions_total = Counter(
    'ml_predictions_total',
    'Total number of predictions made',
    ['model_version', 'class_predicted', 'status'],
    registry=registry
)

# Gauge : accuracy courante (peut monter et descendre)
model_accuracy = Gauge(
    'ml_model_accuracy',
    'Current model accuracy on recent predictions',
    ['model_version'],
    registry=registry
)

# Gauge : drift p-value par feature
feature_drift_pvalue = Gauge(
    'ml_feature_drift_pvalue',
    'KS test p-value for data drift detection',
    ['feature_name'],
    registry=registry
)

# Histogram : latence des prédictions
prediction_latency = Histogram(
    'ml_prediction_latency_seconds',
    'Prediction latency in seconds',
    buckets=(0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0),
    registry=registry
)

# Gauge : nombre de requêtes actives
active_requests = Gauge(
    'ml_active_requests',
    'Number of currently processing requests',
    registry=registry
)

print("✅ Métriques Prometheus définies :")
print("   - ml_predictions_total      (Counter)")
print("   - ml_model_accuracy         (Gauge)")
print("   - ml_feature_drift_pvalue   (Gauge)")
print("   - ml_prediction_latency_s   (Histogram)")
print("   - ml_active_requests        (Gauge)")

In [ ]:
# ============================================================
# SIMULER 100 PRÉDICTIONS ET METTRE À JOUR LES MÉTRIQUES
# ============================================================

# Entraîner un modèle
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED
)
model = RandomForestClassifier(n_estimators=50, random_state=SEED)
model.fit(X_train, y_train)

iris_names = load_iris().target_names

print("🔄 Simulation de 100 prédictions en production...")
print()

N_PREDICTIONS = 100
latencies = []
y_true_log, y_pred_log = [], []

for i in range(N_PREDICTIONS):
    # Prendre un échantillon aléatoire du test set
    idx = np.random.randint(0, len(X_test))
    features = X_test[idx]
    true_label = y_test[idx]

    active_requests.inc()

    # Mesurer la latence
    start = time.time()
    pred = model.predict([features])[0]
    latency = time.time() - start

    # Simuler une latence plus réaliste
    simulated_latency = latency + np.random.exponential(0.01)
    latencies.append(simulated_latency)

    # Mettre à jour les métriques
    prediction_latency.observe(simulated_latency)
    predictions_total.labels(
        model_version='1.0',
        class_predicted=iris_names[pred],
        status='success'
    ).inc()

    active_requests.dec()

    y_true_log.append(true_label)
    y_pred_log.append(pred)

    # Mettre à jour l'accuracy toutes les 20 prédictions
    if (i + 1) % 20 == 0:
        current_acc = accuracy_score(y_true_log, y_pred_log)
        model_accuracy.labels(model_version='1.0').set(current_acc)

# Mettre à jour le drift p-value pour chaque feature
for j, fname in enumerate(load_iris().feature_names):
    _, p_val = ks_2samp(X_train[:, j], X_test[:, j])
    feature_drift_pvalue.labels(feature_name=fname.replace(' ', '_')).set(p_val)

final_accuracy = accuracy_score(y_true_log, y_pred_log)

print(f"  ✅ {N_PREDICTIONS} prédictions effectuées")
print(f"  📊 Accuracy finale     : {final_accuracy:.4f}")
print(f"  ⏱️  Latence moyenne    : {np.mean(latencies)*1000:.2f} ms")
print(f"  ⏱️  Latence P95        : {np.percentile(latencies, 95)*1000:.2f} ms")
print(f"  ⏱️  Latence P99        : {np.percentile(latencies, 99)*1000:.2f} ms")

In [ ]:
# ============================================================
# AFFICHER LES MÉTRIQUES AU FORMAT PROMETHEUS
# ============================================================

metrics_output = generate_latest(registry).decode('utf-8')

print("📋 Métriques au format Prometheus (extrait) :")
print("-" * 65)

# Afficher un extrait pertinent
lines = metrics_output.split('\n')
relevant_lines = [l for l in lines if not l.startswith('#') and l.strip()]
for line in relevant_lines[:20]:
    print(f"  {line}")

print("-" * 65)
print()
print("📋 Pour exposer ces métriques en production :")
print("   from prometheus_client import start_http_server")
print("   start_http_server(8000)  # → http://localhost:8000/metrics")
print("   # Prometheus scrape toutes les 30s automatiquement")

In [ ]:
# ============================================================
# SIMULER UN DASHBOARD DE MONITORING
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 Dashboard Monitoring ML — TP 3.2', fontsize=14, fontweight='bold')

# ---- 1. Accuracy dans le temps ----
window_size = 10
rolling_acc = [accuracy_score(y_true_log[max(0,i-window_size):i+1],
                               y_pred_log[max(0,i-window_size):i+1])
               for i in range(1, N_PREDICTIONS)]

axes[0,0].plot(rolling_acc, color='steelblue', linewidth=2, label='Accuracy (rolling 10)')
axes[0,0].axhline(0.85, color='red',    linestyle='--', linewidth=1.5, label='Seuil CRITICAL (0.85)')
axes[0,0].axhline(0.90, color='orange', linestyle='--', linewidth=1.5, label='Seuil WARNING (0.90)')
axes[0,0].set_title('📈 Accuracy dans le temps')
axes[0,0].set_xlabel('Prédiction #')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_ylim(0.5, 1.05)
axes[0,0].legend(fontsize=8)
axes[0,0].grid(alpha=0.3)

# ---- 2. Distribution des latences ----
axes[0,1].hist([l*1000 for l in latencies], bins=30, color='coral', alpha=0.8, edgecolor='white')
axes[0,1].axvline(np.mean(latencies)*1000, color='red',    linestyle='--', linewidth=2, label=f'Mean: {np.mean(latencies)*1000:.1f}ms')
axes[0,1].axvline(np.percentile(latencies,95)*1000, color='orange', linestyle='--', linewidth=2, label=f'P95: {np.percentile(latencies,95)*1000:.1f}ms')
axes[0,1].axvline(500, color='darkred', linestyle='-', linewidth=1.5, alpha=0.7, label='SLA: 500ms')
axes[0,1].set_title('⏱️ Distribution des latences')
axes[0,1].set_xlabel('Latence (ms)')
axes[0,1].set_ylabel('Nombre de requêtes')
axes[0,1].legend(fontsize=8)
axes[0,1].grid(alpha=0.3)

# ---- 3. Distribution des classes prédites ----
from collections import Counter
pred_counts = Counter([iris_names[p] for p in y_pred_log])
true_counts  = Counter([iris_names[t] for t in y_true_log])
classes_names = list(pred_counts.keys())
x = np.arange(len(classes_names))
axes[1,0].bar(x - 0.2, [pred_counts.get(c,0) for c in classes_names], 0.4, label='Prédictions', color='steelblue', alpha=0.8)
axes[1,0].bar(x + 0.2, [true_counts.get(c,0) for c in classes_names],  0.4, label='Réel',         color='coral', alpha=0.8)
axes[1,0].set_title('📊 Distribution des classes')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(classes_names)
axes[1,0].set_ylabel('Nombre')
axes[1,0].legend()
axes[1,0].grid(axis='y', alpha=0.3)

# ---- 4. Drift p-values par feature ----
fnames_short = ['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid']
p_values_plot = []
for j in range(4):
    _, pv = ks_2samp(X_train[:, j], X_test[:, j])
    p_values_plot.append(pv)

bar_colors = ['green' if pv > 0.05 else 'orange' if pv > 0.01 else 'red' for pv in p_values_plot]
axes[1,1].bar(fnames_short, p_values_plot, color=bar_colors, alpha=0.8, edgecolor='white')
axes[1,1].axhline(0.05, color='orange', linestyle='--', linewidth=2, label='Seuil drift (0.05)')
axes[1,1].axhline(0.01, color='red',    linestyle='--', linewidth=2, label='Seuil critique (0.01)')
axes[1,1].set_title('🌊 Drift Detection — p-values')
axes[1,1].set_ylabel('p-value (KS test)')
axes[1,1].set_ylim(0, 1)
axes[1,1].legend(fontsize=8)
axes[1,1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('monitoring_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Dashboard sauvegardé : monitoring_dashboard.png")

<a id='7'></a>
## 7. 🔬 A/B Testing pour Modèles ML

Avant de déployer un nouveau modèle à **100% du trafic**, on le teste sur un petit pourcentage d'utilisateurs.

### 3 Stratégies de déploiement

```
┌──────────────────────────────────────────────────────────┐
│  ALL-OR-NOTHING  │  0% ──────────────────────────► 100%  │
│                  │  Simple mais risqué                    │
├──────────────────┼───────────────────────────────────────┤
│  CANARY          │  0% ► 5% ► 20% ► 50% ► 100%          │
│                  │  Progressif, détecte les problèmes tôt │
├──────────────────┼───────────────────────────────────────┤
│  A/B TEST        │  50% → Modèle A   50% → Modèle B      │
│                  │  Compare statistiquement les 2         │
└──────────────────┴───────────────────────────────────────┘
```

### Méthode statistique : Test Chi-Square

Pour savoir si la différence entre A et B est **réelle** ou due au **hasard** :

- **H0** (hypothèse nulle) : pas de différence entre A et B
- **H1** (hypothèse alternative) : B est significativement meilleur que A
- Si **p-value < 0.05** → rejeter H0 → la différence est réelle

> 🛒 **Amazon** : Amazon teste continuellement ses algorithmes de recommandation. Une amélioration de **0.1% en conversion** = **millions $** de revenu additionnel sur un an.

<a id='8'></a>
## 8. 🧪 TP 3.3 — A/B Test (Chi-Square)

In [ ]:
# ============================================================
# TP 3.3 — A/B TEST STATISTIQUE
# ============================================================

def ab_test(group_a_conversions: int, group_a_total: int,
            group_b_conversions: int, group_b_total: int,
            significance_level: float = 0.05) -> dict:
    """
    Test statistique Chi-Square pour comparer deux modèles ML.

    Args:
        group_a_conversions : Nombre de bonnes prédictions du modèle A
        group_a_total       : Nombre total de prédictions du modèle A
        group_b_conversions : Nombre de bonnes prédictions du modèle B
        group_b_total       : Nombre total de prédictions du modèle B
        significance_level  : Seuil de significativité (défaut 0.05)

    Returns:
        dict avec résultats, significativité et recommandation
    """
    # Table de contingence
    # [succès_A, échecs_A]
    # [succès_B, échecs_B]
    contingency_table = [
        [group_a_conversions, group_a_total - group_a_conversions],
        [group_b_conversions, group_b_total - group_b_conversions]
    ]

    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    conv_a = group_a_conversions / group_a_total
    conv_b = group_b_conversions / group_b_total
    lift   = (conv_b - conv_a) / conv_a * 100 if conv_a > 0 else 0

    is_significant = p_value < significance_level

    if is_significant and conv_b > conv_a:
        recommendation = "✅ Déployer le Modèle B en production"
        winner         = "Modèle B"
    elif is_significant and conv_a > conv_b:
        recommendation = "⚠️ Garder le Modèle A (B est moins bon)"
        winner         = "Modèle A"
    else:
        recommendation = "🔄 Pas assez de données / différence non significative"
        winner         = "Indécis"

    return {
        "model_a_accuracy"  : round(conv_a, 4),
        "model_b_accuracy"  : round(conv_b, 4),
        "lift"              : round(lift, 2),
        "chi2_statistic"    : round(float(chi2), 4),
        "p_value"           : round(float(p_value), 6),
        "degrees_of_freedom": dof,
        "is_significant"    : is_significant,
        "winner"            : winner,
        "recommendation"    : recommendation
    }

print("✅ Fonction ab_test définie !")

In [ ]:
# ============================================================
# SCÉNARIOS A/B TEST — 3 CAS DIFFÉRENTS
# ============================================================

ab_scenarios = [
    {
        "name"  : "Scénario 1 — Amélioration significative",
        "a_ok"  : 8500,  "a_total": 10000,   # Modèle A : 85%
        "b_ok"  : 9200,  "b_total": 10000,   # Modèle B : 92%
        "desc"  : "Nouveau modèle clairement meilleur"
    },
    {
        "name"  : "Scénario 2 — Légère amélioration",
        "a_ok"  : 8500,  "a_total": 10000,   # Modèle A : 85%
        "b_ok"  : 8560,  "b_total": 10000,   # Modèle B : 85.6%
        "desc"  : "Légère différence, peut-être due au hasard"
    },
    {
        "name"  : "Scénario 3 — Modèle B dégradé",
        "a_ok"  : 8500,  "a_total": 10000,   # Modèle A : 85%
        "b_ok"  : 7800,  "b_total": 10000,   # Modèle B : 78%
        "desc"  : "Nouveau modèle moins bon → garder l\'ancien"
    }
]

ab_results = []
for sc in ab_scenarios:
    res = ab_test(sc["a_ok"], sc["a_total"], sc["b_ok"], sc["b_total"])
    res["scenario"] = sc["name"]
    ab_results.append(res)

    print("=" * 65)
    print(f"  📌 {sc['name']}")
    print(f"     {sc['desc']}")
    print(f"     Modèle A accuracy : {res['model_a_accuracy']*100:.1f}%  ({sc['a_ok']:,}/{sc['a_total']:,})")
    print(f"     Modèle B accuracy : {res['model_b_accuracy']*100:.1f}%  ({sc['b_ok']:,}/{sc['b_total']:,})")
    print(f"     Lift              : {res['lift']:+.2f}%")
    print(f"     p-value           : {res['p_value']:.6f}  {'(< 0.05 → significatif)' if res['is_significant'] else '(> 0.05 → non significatif)'}")
    print(f"     Gagnant           : {res['winner']}")
    print(f"     👉 {res['recommendation']}")
    print()

In [ ]:
# ============================================================
# A/B TEST RÉEL : COMPARER 2 MODÈLES ML SUR IRIS
# ============================================================

print("🔬 A/B Test réel : RandomForest vs GradientBoosting")
print("=" * 65)

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=SEED
)

# Modèle A (production actuelle)
model_a = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=SEED)
model_a.fit(X_train, y_train)
pred_a = model_a.predict(X_test)
correct_a = int(np.sum(pred_a == y_test))

# Modèle B (candidat au déploiement)
model_b = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED)
model_b.fit(X_train, y_train)
pred_b = model_b.predict(X_test)
correct_b = int(np.sum(pred_b == y_test))

print(f"  Modèle A (Random Forest)        : {correct_a}/{len(X_test)} correctes ({correct_a/len(X_test)*100:.1f}%)")
print(f"  Modèle B (Gradient Boosting)    : {correct_b}/{len(X_test)} correctes ({correct_b/len(X_test)*100:.1f}%)")
print()

real_ab = ab_test(correct_a, len(X_test), correct_b, len(X_test))

print(f"  Chi2 statistic : {real_ab['chi2_statistic']}")
print(f"  p-value        : {real_ab['p_value']}")
print(f"  Significatif   : {'OUI' if real_ab['is_significant'] else 'NON (besoin de plus de données)'}")
print(f"  Lift           : {real_ab['lift']:+.2f}%")
print(f"  Gagnant        : {real_ab['winner']}")
print(f"  👉 {real_ab['recommendation']}")

print()
if not real_ab['is_significant']:
    print("  💡 Le dataset Iris est petit (60 samples de test).")
    print("     En production, on aurait des milliers d'observations")
    print("     → la différence deviendrait significative.")

<a id='9'></a>
## 9. ⚖️ Gouvernance — Explicabilité avec SHAP

Les réglementations (**RGPD** en Europe, **FTC** aux USA) exigent que les modèles ML soient **explicables** et **justes**.

### Droits RGPD liés aux modèles ML

| Droit | Implication ML | Solution |
|---|---|---|
| **Droit à l'explication** | Expliquer pourquoi le crédit est refusé | SHAP / LIME |
| **Droit à la rectification** | Corriger les données erronées | Audit trail |
| **Non-discrimination** | Le modèle ne doit pas discriminer | Fairness testing |
| **Traçabilité** | Savoir qui a décidé quoi | MLflow Registry |

### SHAP (SHapley Additive exPlanations)

SHAP répond à : **"Pourquoi le modèle a-t-il prédit ça ?"**

```
Exemple — Prédiction de crédit :

Base value (moyenne) : 0.0

+ Age (35 ans)       : +0.10  ✅ bon signe
- Revenu (18K€)      : -0.45  ❌ trop bas
- Score crédit (380) : -0.30  ❌ mauvais
+ Historique (5 ans) : +0.05  ✅ positif
─────────────────────────────
Prédiction finale   : -0.60  → REFUS
```

> ⚖️ **Régulation RGPD** : Une banque qui refuse un crédit avec une IA **DOIT** pouvoir expliquer pourquoi à l'utilisateur. Sans explicabilité, c'est **illégal**.

In [ ]:
# ============================================================
# EXPLICABILITÉ — IMPLÉMENTATION MANUELLE (sans SHAP)
# (Feature Importance comme proxy d'explicabilité)
# ============================================================

X, y = load_iris(return_X_y=True)
feature_names = load_iris().feature_names
target_names  = load_iris().target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

model_xai = RandomForestClassifier(n_estimators=100, random_state=SEED)
model_xai.fit(X_train, y_train)

# ---- Feature importance globale ----
importances = model_xai.feature_importances_
importance_df = pd.DataFrame({
    'feature'   : feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("📊 Feature Importance Globale (modèle complet) :")
print("=" * 50)
for _, row in importance_df.iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f"  {row['feature']:30s} {row['importance']:.4f}  {bar}")
print("=" * 50)
print()

# ---- Explication d'une prédiction individuelle ----
sample_idx = 5
sample = X_test[sample_idx]
pred   = model_xai.predict([sample])[0]
proba  = model_xai.predict_proba([sample])[0]

print(f"🔍 Explication de la prédiction #{sample_idx} :")
print(f"   Prédiction : {target_names[pred]} (confiance={proba[pred]*100:.1f}%)")
print(f"   Réel       : {target_names[y_test[sample_idx]]}")
print()
print("   Features et leur influence estimée :")
for fname, fval, fimp in zip(feature_names, sample, importances):
    direction = "+" if fval > np.mean(X_train[:, list(feature_names).index(fname)]) else "-"
    print(f"   {fname:30s} = {fval:.2f}  (importance: {fimp:.4f}, direction: {direction})")

In [ ]:
# ============================================================
# VISUALISATION EXPLICABILITÉ
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('⚖️ Explicabilité du Modèle — Gouvernance RGPD', fontsize=13, fontweight='bold')

# ---- 1. Feature Importance globale ----
colors_bar = ['#2ecc71' if i == 0 else '#3498db' if i == 1 else '#e74c3c' for i in range(len(feature_names))]
bars = axes[0].barh(importance_df['feature'], importance_df['importance'],
                     color=colors_bar[::-1], alpha=0.85, edgecolor='white')
axes[0].set_title('📊 Feature Importance Globale\n(Random Forest)')
axes[0].set_xlabel('Importance')
axes[0].grid(axis='x', alpha=0.3)
for bar, val in zip(bars, importance_df['importance'].values[::-1]):
    axes[0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)

# ---- 2. Contributions par prédiction (waterfall simplifié) ----
sample     = X_test[sample_idx]
base_value = np.mean(y_train == pred)
contributions = []
for j, (fname, fval, fimp) in enumerate(zip(feature_names, sample, importances)):
    mean_val = np.mean(X_train[:, j])
    direction = 1 if fval > mean_val else -1
    contrib = direction * fimp
    contributions.append((fname.replace(' (cm)', ''), contrib))

contributions.sort(key=lambda x: x[1])
fnames_c  = [c[0] for c in contributions]
contribs  = [c[1] for c in contributions]
bar_colors_wf = ['#2ecc71' if c > 0 else '#e74c3c' for c in contribs]

axes[1].barh(fnames_c, contribs, color=bar_colors_wf, alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title(f'🔍 Explication locale — Sample #{sample_idx}\nPrédiction: {target_names[pred]}')
axes[1].set_xlabel('Contribution (positif = pousse vers la prédiction)')
axes[1].grid(axis='x', alpha=0.3)

green_patch = mpatches.Patch(color='#2ecc71', label='Positif (augmente la confiance)')
red_patch   = mpatches.Patch(color='#e74c3c', label='Négatif (réduit la confiance)')
axes[1].legend(handles=[green_patch, red_patch], fontsize=8)

plt.tight_layout()
plt.savefig('explicabilite_shap.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Graphique sauvegardé : explicabilite_shap.png")

<a id='10'></a>
## 10. 🧪 TP 3.4 — Fairness & Détection de Biais

### Objectif
Détecter si un modèle **discrimine** certains groupes (genre, âge, origine) dans ses prédictions.

### Définition du biais ML

Un modèle est **biaisé** si ses prédictions traitent différemment des groupes démographiques **de façon injustifiée**.

| Sévérité | Différence | Action |
|---|---|---|
| NORMAL ✅ | < 10% | Aucune |
| WARNING ⚠️ | 10% – 20% | Analyser |
| CRITICAL 🚨 | > 20% | Retraining obligatoire |

> 🏢 **Amazon (Hiring)** : Amazon a découvert que son modèle de recrutement était **biaisé contre les femmes** (entraîné sur données historiques où les hommes dominaient l'ingénierie). Amazon a dû **rejeter le modèle**. Le fairness testing est maintenant obligatoire.

In [ ]:
# ============================================================
# TP 3.4 — FAIRNESS & DÉTECTION DE BIAIS
# ============================================================

def detect_bias(predictions: np.ndarray,
                protected_attribute: np.ndarray,
                protected_value,
                group_a_name: str = "Groupe protégé",
                group_b_name: str = "Groupe de référence") -> dict:
    """
    Détecte si un modèle traite différemment deux groupes.

    Args:
        predictions         : Prédictions du modèle
        protected_attribute : Attribut protégé (genre, âge...)
        protected_value     : Valeur du groupe protégé (ex: 'F')
        group_a_name        : Nom du groupe protégé
        group_b_name        : Nom du groupe de référence

    Returns:
        dict avec statistiques et niveau de biais
    """
    mask_a = protected_attribute == protected_value
    preds_a = predictions[mask_a]
    preds_b = predictions[~mask_a]

    mean_a = float(np.mean(preds_a))
    mean_b = float(np.mean(preds_b))
    diff   = abs(mean_a - mean_b)
    ratio  = mean_a / mean_b if mean_b != 0 else 0

    # Test statistique
    from scipy.stats import mannwhitneyu
    _, pvalue_stat = mannwhitneyu(preds_a, preds_b, alternative='two-sided')

    if diff > 0.20:
        severity       = "CRITICAL 🚨"
        recommendation = "Retraining obligatoire avec données équilibrées"
    elif diff > 0.10:
        severity       = "WARNING ⚠️"
        recommendation = "Analyser la source du biais"
    else:
        severity       = "NORMAL ✅"
        recommendation = "Modèle équitable"

    return {
        "group_a"        : {"name": group_a_name, "mean": round(mean_a, 4), "count": int(sum(mask_a))},
        "group_b"        : {"name": group_b_name, "mean": round(mean_b, 4), "count": int(sum(~mask_a))},
        "difference"     : round(diff, 4),
        "ratio"          : round(ratio, 4),
        "p_value"        : round(float(pvalue_stat), 6),
        "is_significant" : pvalue_stat < 0.05,
        "severity"       : severity,
        "recommendation" : recommendation
    }

print("✅ Fonction detect_bias définie !")

In [ ]:
# ============================================================
# SIMULATION — PRÉDICTIONS DE SALAIRE AVEC BIAIS
# ============================================================

np.random.seed(SEED)
N = 1000

# Créer un dataset de prédiction de salaire
# Simuler 3 scénarios de biais différents
fairness_scenarios = [
    {
        "name"          : "Scénario 1 — Modèle équitable",
        "pred_women"    : np.random.normal(52000, 8000, N//2),
        "pred_men"      : np.random.normal(53000, 8000, N//2),
        "desc"          : "Différence ≈ 1K€ (normal)"
    },
    {
        "name"          : "Scénario 2 — Biais modéré",
        "pred_women"    : np.random.normal(48000, 8000, N//2),
        "pred_men"      : np.random.normal(55000, 8000, N//2),
        "desc"          : "Femmes sous-estimées de ~7K€"
    },
    {
        "name"          : "Scénario 3 — Biais critique",
        "pred_women"    : np.random.normal(42000, 8000, N//2),
        "pred_men"      : np.random.normal(65000, 8000, N//2),
        "desc"          : "Femmes sous-estimées de ~23K€ ← ILLÉGAL"
    }
]

for sc in fairness_scenarios:
    all_preds  = np.concatenate([sc["pred_women"], sc["pred_men"]])
    all_gender = np.array(['F'] * (N//2) + ['M'] * (N//2))

    # Normaliser pour le test (0 à 1)
    norm_preds = (all_preds - all_preds.min()) / (all_preds.max() - all_preds.min())

    res = detect_bias(norm_preds, all_gender, 'F',
                      group_a_name='Femmes', group_b_name='Hommes')

    diff_salary = abs(np.mean(sc["pred_women"]) - np.mean(sc["pred_men"]))

    print("=" * 65)
    print(f"  📌 {sc['name']}")
    print(f"     {sc['desc']}")
    print(f"     Salaire moyen femmes  : {np.mean(sc['pred_women']):,.0f}€")
    print(f"     Salaire moyen hommes  : {np.mean(sc['pred_men']):,.0f}€")
    print(f"     Écart salarial        : {diff_salary:,.0f}€")
    print(f"     Différence normalisée : {res['difference']:.4f}")
    print(f"     Significatif (stat.)  : {'OUI' if res['is_significant'] else 'NON'}")
    print(f"     Sévérité              : {res['severity']}")
    print(f"     👉 {res['recommendation']}")
    print()

In [ ]:
# ============================================================
# VISUALISATION DES 3 SCÉNARIOS DE BIAIS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('⚖️ Détection de Biais — Prédictions de Salaire — TP 3.4',
             fontsize=13, fontweight='bold')

severity_colors = {"NORMAL ✅": "green", "WARNING ⚠️": "orange", "CRITICAL 🚨": "red"}

for i, sc in enumerate(fairness_scenarios):
    ax = axes[i]

    all_preds  = np.concatenate([sc["pred_women"], sc["pred_men"]])
    all_gender = np.array(['F'] * (N//2) + ['M'] * (N//2))
    norm_preds = (all_preds - all_preds.min()) / (all_preds.max() - all_preds.min())
    res = detect_bias(norm_preds, all_gender, 'F', 'Femmes', 'Hommes')

    diff_k = abs(np.mean(sc["pred_women"]) - np.mean(sc["pred_men"])) / 1000
    border_color = severity_colors.get(res["severity"], "gray")

    ax.hist(sc["pred_women"]/1000, bins=30, alpha=0.65, color='#e91e8c',
            label=f'Femmes (moy: {np.mean(sc["pred_women"])/1000:.0f}K€)', density=True)
    ax.hist(sc["pred_men"]/1000,   bins=30, alpha=0.65, color='#2196F3',
            label=f'Hommes (moy: {np.mean(sc["pred_men"])/1000:.0f}K€)',   density=True)

    ax.axvline(np.mean(sc["pred_women"])/1000, color='#e91e8c', linestyle='--', linewidth=2)
    ax.axvline(np.mean(sc["pred_men"])/1000,   color='#2196F3', linestyle='--', linewidth=2)

    ax.set_title(f'{sc["name"]}\nÉcart: {diff_k:.0f}K€ | {res["severity"]}', fontsize=9)
    ax.set_xlabel('Salaire prédit (K€)')
    ax.set_ylabel('Densité')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)

plt.tight_layout()
plt.savefig('fairness_bias_detection.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Graphique sauvegardé : fairness_bias_detection.png")

<a id='11'></a>
## 11. 📝 Résumé et Bilan du Jour 3

### ✅ Ce que tu as fait aujourd'hui

| Concept | Ce que tu as réalisé |
|---|---|
| **Drift Detection** | KS test sur 3 niveaux + toutes les features Iris |
| **Prometheus** | 5 métriques ML + simulation 100 prédictions |
| **Dashboard** | 4 graphiques : accuracy, latence, classes, drift |
| **A/B Testing** | Chi-square sur 3 scénarios + test réel RF vs GB |
| **Explicabilité** | Feature importance globale + explication locale |
| **Fairness** | 3 scénarios de biais de salaire (NORMAL/WARNING/CRITICAL) |

### 🎯 Points clés

1. **Data drift** et **concept drift** dégradent les modèles en production naturellement
2. **KS test** : p-value < 0.05 → drift confirmé → retraining requis
3. **Prometheus + Grafana** détectent les problèmes **avant** les utilisateurs
4. **A/B testing** valide que les améliorations sont **statistiquement réelles**
5. **SHAP + Fairness** sont **obligatoires légalement** (RGPD)
6. **Gouvernance** = reproductibilité + auditabilité + conformité

### 🚀 Demain — Jour 4
**Natural Language Processing** :  
→ Tokenisation, BERT, fine-tuning, Hugging Face, NER, sentiment analysis

---

### 📁 Fichiers générés
```
jour3/
├── jour3_monitoring_gouvernance.ipynb
├── drift_detection_visualization.png
├── monitoring_dashboard.png
├── explicabilite_shap.png
└── fairness_bias_detection.png
```

In [ ]:
# ============================================================
# ✅ VÉRIFICATION FINALE — Bilan du Jour 3
# ============================================================
import os

print("=" * 65)
print("         🎉 BILAN JOUR 3 — MONITORING & GOUVERNANCE")
print("=" * 65)
print()

checks = [
    ("TP 3.1 — Drift Detection (KS test)",   len(drift_results) == 3),
    ("TP 3.1 — Drift sur toutes features",   len(feature_drift_results) == 4),
    ("TP 3.2 — Métriques Prometheus",        True),
    ("TP 3.2 — 100 prédictions simulées",    len(latencies) == 100),
    ("TP 3.3 — A/B Test Chi-Square",         len(ab_results) == 3),
    ("TP 3.3 — A/B Test réel RF vs GB",      real_ab is not None),
    ("TP 3.4 — Fairness 3 scénarios",        True),
    ("Graphique drift",                      os.path.exists('drift_detection_visualization.png')),
    ("Dashboard monitoring",                 os.path.exists('monitoring_dashboard.png')),
    ("Graphique explicabilité",              os.path.exists('explicabilite_shap.png')),
    ("Graphique fairness",                   os.path.exists('fairness_bias_detection.png')),
]

all_ok = True
for label, condition in checks:
    status = "✅" if condition else "❌"
    if not condition:
        all_ok = False
    print(f"  {status} {label}")

print()
print("=" * 65)
if all_ok:
    print("  🏆 Tous les TPs du Jour 3 sont COMPLÉTÉS !")
else:
    print("  ⚠️  Certains éléments manquent, relance les cellules.")
print("=" * 65)
print()
print("  📌 Git commit suggéré :")
print("  git add jour3/")
print('  git commit -m "✅ Jour 3 - Monitoring + Drift + A/B Test + Gouvernance"')
print("  git push")
print("=" * 65)